# Customer Segmentation Analysis using RFM and K-Means Clustering

## Objective
Phân khúc khách hàng dựa trên hành vi mua hàng sử dụng RFM metrics và K-Means clustering.

Phân tích này giúp:
- Xác định khách hàng giá trị cao (VIP)
- Phát hiện khách hàng có nguy cơ rời đi
- Cải thiện tỷ lệ giữ chân khách hàng
- Tối ưu hóa chiến lược marketing theo từng phân khúc

### Các sửa đổi so với phiên bản cũ:
- **[FIX]** Sửa cột Monetary từ `Invoice` → `total_revenue` (doanh thu thực)
- **[FIX]** Dynamic cluster naming dựa trên profile thực tế, không hard-code
- **[ADD]** Log-transform Monetary trước khi scale (xử lý phân phối lệch)
- **[ADD]** Silhouette Score + Davies-Bouldin Index để validate cluster
- **[ADD]** Heuristic segment cải tiến theo từng chiều R/F/M riêng biệt
- **[ADD]** Business action recommendations per segment
- **[ADD]** Outlier detection trước khi clustering
- **[ADD]** Export kết quả ra CSV để dùng cho CLV và Churn notebooks

In [ ]:
# =========================================================
# IMPORT LIBRARIES
# =========================================================
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, davies_bouldin_score

import warnings
warnings.filterwarnings('ignore')

# Plot style
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.dpi'] = 100

## 1. Load Dataset

In [ ]:
# Load processed dataset
df = pd.read_csv("Dataset/Cleaned_data/master_df.csv")

# Convert Transaction_Date to datetime
df["Transaction_Date"] = pd.to_datetime(df["Transaction_Date"])

print(f"Dataset shape: {df.shape}")
print(f"Date range: {df['Transaction_Date'].min().date()} → {df['Transaction_Date'].max().date()}")
print(f"Unique customers: {df['CustomerID'].nunique():,}")
print(f"\nColumns available: {list(df.columns)}")
df.head()

## 2. RFM Analysis

RFM đo lường giá trị khách hàng qua 3 chỉ số:
- **Recency**: Số ngày kể từ lần mua hàng gần nhất (càng nhỏ càng tốt)
- **Frequency**: Số giao dịch unique (càng cao càng tốt)
- **Monetary**: Tổng doanh thu (càng cao càng tốt)

> **[FIX v2]** Monetary dùng cột `total_revenue` thay vì `Invoice`. `Invoice` là mã đơn hàng (ID), không phải giá trị tài chính.

In [ ]:
# =========================================================
# [FIX] XÁC NHẬN CỘT DOANH THU ĐÚNG
# =========================================================
# Tự động detect cột revenue phù hợp
revenue_candidates = ['total_revenue', 'revenue', 'Total_Revenue',
                      'amount', 'Amount', 'sales', 'Sales']
monetary_col = None
for col in revenue_candidates:
    if col in df.columns:
        monetary_col = col
        break

if monetary_col is None:
    raise ValueError(
        f"Không tìm thấy cột doanh thu. "
        f"Các cột hiện có: {list(df.columns)}. "
        f"Vui lòng cập nhật revenue_candidates."
    )

print(f"✓ Monetary column: '{monetary_col}'")
print(f"  Sample values: {df[monetary_col].describe().round(2).to_dict()}")

In [ ]:
# =========================================================
# TẠO BẢNG RFM
# =========================================================
today = df["Transaction_Date"].max() + pd.Timedelta(days=1)

rfm = df.groupby("CustomerID").agg(
    Recency=("Transaction_Date",
             lambda x: (today - x.max()).days),
    Frequency=("Transaction_ID", "nunique"),
    Monetary=(monetary_col, "sum")
).reset_index()

# Lọc bỏ Monetary <= 0 (giao dịch hoàn trả thuần)
n_before = len(rfm)
rfm = rfm[rfm["Monetary"] > 0].copy()
print(f"Loại bỏ {n_before - len(rfm)} khách hàng có Monetary <= 0")
print(f"\nRFM table: {len(rfm):,} customers")
print(rfm[["Recency","Frequency","Monetary"]].describe().round(2))

## 3. Kiểm tra Outlier

> **[ADD v2]** Outlier có Monetary cực cao sẽ kéo lệch centroid K-Means. Dùng IQR method để phát hiện và log-transform để giảm ảnh hưởng.

In [ ]:
# =========================================================
# OUTLIER DETECTION (IQR METHOD)
# =========================================================
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

for ax, col in zip(axes, ["Recency", "Frequency", "Monetary"]):
    ax.boxplot(rfm[col], vert=True, patch_artist=True,
               boxprops=dict(facecolor='steelblue', alpha=0.6))
    ax.set_title(col)
    ax.set_ylabel("Value")

plt.suptitle("Outlier Check — RFM Distributions", fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

# Đếm outlier
for col in ["Recency", "Frequency", "Monetary"]:
    Q1 = rfm[col].quantile(0.25)
    Q3 = rfm[col].quantile(0.75)
    IQR = Q3 - Q1
    outliers = rfm[(rfm[col] < Q1 - 1.5*IQR) | (rfm[col] > Q3 + 1.5*IQR)]
    print(f"{col}: {len(outliers)} outliers ({len(outliers)/len(rfm)*100:.1f}%)")

In [ ]:
# =========================================================
# [ADD] LOG-TRANSFORM MONETARY
# Monetary thường có phân phối lệch phải mạnh.
# Log-transform giúp StandardScaler hoạt động tốt hơn.
# =========================================================
rfm["Monetary_Log"] = np.log1p(rfm["Monetary"])  # log1p = log(1+x), an toàn khi có 0

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(rfm["Monetary"], bins=50, color='steelblue', edgecolor='white', alpha=0.8)
axes[0].set_title("Monetary — Original (thường lệch phải)")
axes[0].set_xlabel("Revenue")

axes[1].hist(rfm["Monetary_Log"], bins=50, color='teal', edgecolor='white', alpha=0.8)
axes[1].set_title("Monetary — Log-transformed (phân phối đều hơn)")
axes[1].set_xlabel("log(1 + Revenue)")

plt.tight_layout()
plt.show()

## 4. RFM Scoring

Mỗi khách hàng nhận điểm từ 1–4 cho từng chiều R, F, M.

In [ ]:
# =========================================================
# RFM SCORING (1-4)
# =========================================================
rfm["R_Score"] = pd.qcut(
    rfm["Recency"], q=4, labels=[4,3,2,1]
).astype(int)

rfm["F_Score"] = pd.qcut(
    rfm["Frequency"].rank(method="first"),
    q=4, labels=[1,2,3,4]
).astype(int)

rfm["M_Score"] = pd.qcut(
    rfm["Monetary"].rank(method="first"),
    q=4, labels=[1,2,3,4]
).astype(int)

rfm["RFM_Score"] = rfm["R_Score"] + rfm["F_Score"] + rfm["M_Score"]

print("Score distribution:")
print(rfm["RFM_Score"].value_counts().sort_index())

## 5. Heuristic Segmentation (cải tiến)

> **[FIX v2]** Phiên bản cũ chỉ dùng tổng RFM Score, dẫn đến hai khách hàng có hành vi khác nhau nhưng cùng tổng điểm được xếp chung nhóm. Phiên bản mới phân loại theo từng chiều R/F/M riêng biệt.

In [ ]:
# =========================================================
# [FIX] HEURISTIC SEGMENTATION CẢI TIẾN
# Dùng từng chiều R, F, M thay vì chỉ tổng score
# =========================================================
def heuristic_segment(row):
    r, f, m = row["R_Score"], row["F_Score"], row["M_Score"]

    # Champions: mua gần, thường xuyên, và nhiều tiền
    if r >= 4 and f >= 4 and m >= 4:
        return "Champions"

    # Loyal: mua thường xuyên và nhiều tiền, dù không quá gần đây
    if f >= 3 and m >= 3:
        return "Loyal Customers"

    # Potential Loyalists: mua gần đây, có tần suất trung bình
    if r >= 3 and f >= 2:
        return "Potential Loyalists"

    # At Risk: từng mua nhiều nhưng lâu rồi không quay lại
    if r <= 2 and f >= 3:
        return "At Risk"

    # Hibernating: mua ít, lâu không quay lại
    if r <= 2 and f <= 2:
        return "Hibernating"

    # New Customers: mua gần đây nhưng ít giao dịch
    if r >= 3 and f == 1:
        return "New Customers"

    return "Standard"

rfm["Heuristic_Segment"] = rfm.apply(heuristic_segment, axis=1)

seg_dist = rfm["Heuristic_Segment"].value_counts()
print("Heuristic Segment Distribution:")
print(seg_dist.to_string())
print(f"\nTotal: {seg_dist.sum():,} customers")

In [ ]:
# Visualize heuristic segments
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
seg_dist.plot(kind="bar", ax=axes[0], color="steelblue", edgecolor="white", alpha=0.85)
axes[0].set_title("Heuristic Segment Distribution", fontsize=12)
axes[0].set_xlabel("Segment")
axes[0].set_ylabel("Number of Customers")
axes[0].tick_params(axis='x', rotation=20)

for bar in axes[0].patches:
    axes[0].text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 0.5,
        f"{int(bar.get_height()):,}",
        ha='center', va='bottom', fontsize=9
    )

# Pie chart
axes[1].pie(
    seg_dist.values,
    labels=seg_dist.index,
    autopct="%1.1f%%",
    startangle=140
)
axes[1].set_title("Heuristic Segment Share", fontsize=12)

plt.tight_layout()
plt.show()

## 6. Standardization (dùng Monetary_Log)

> **[FIX v2]** Scale dữ liệu dùng `Monetary_Log` thay vì `Monetary` raw để tránh outlier chi phối clustering.

In [ ]:
# =========================================================
# STANDARDIZATION
# =========================================================
kmeans_features = rfm[["Recency", "Frequency", "Monetary_Log"]]

scaler = StandardScaler()
kmeans_scaled = scaler.fit_transform(kmeans_features)

print("Scaled data stats (should be ~mean=0, std=1 per column):")
print(pd.DataFrame(kmeans_scaled,
                   columns=["Recency","Frequency","Monetary_Log"]
                  ).describe().round(3))

## 7. Xác định số cluster tối ưu

> **[ADD v2]** Kết hợp Elbow Method + Silhouette Score để đưa ra quyết định k có căn cứ.

In [ ]:
# =========================================================
# [ADD] ELBOW + SILHOUETTE ANALYSIS
# =========================================================
k_range = range(2, 10)
wcss = []
silhouette_scores = []
db_scores = []

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(kmeans_scaled)

    wcss.append(km.inertia_)
    silhouette_scores.append(silhouette_score(kmeans_scaled, labels))
    db_scores.append(davies_bouldin_score(kmeans_scaled, labels))

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Elbow
axes[0].plot(k_range, wcss, marker='o', color='steelblue')
axes[0].set_title("Elbow Method (WCSS)")
axes[0].set_xlabel("Number of Clusters (k)")
axes[0].set_ylabel("WCSS")

# Silhouette (higher = better)
best_sil_k = list(k_range)[silhouette_scores.index(max(silhouette_scores))]
axes[1].plot(k_range, silhouette_scores, marker='o', color='teal')
axes[1].axvline(best_sil_k, color='red', linestyle='--', alpha=0.6, label=f"Best k={best_sil_k}")
axes[1].set_title("Silhouette Score (higher = better)")
axes[1].set_xlabel("Number of Clusters (k)")
axes[1].set_ylabel("Silhouette Score")
axes[1].legend()

# Davies-Bouldin (lower = better)
best_db_k = list(k_range)[db_scores.index(min(db_scores))]
axes[2].plot(k_range, db_scores, marker='o', color='coral')
axes[2].axvline(best_db_k, color='red', linestyle='--', alpha=0.6, label=f"Best k={best_db_k}")
axes[2].set_title("Davies-Bouldin Index (lower = better)")
axes[2].set_xlabel("Number of Clusters (k)")
axes[2].set_ylabel("DB Index")
axes[2].legend()

plt.tight_layout()
plt.show()

print("\nCluster validation scores:")
for i, k in enumerate(k_range):
    print(f"  k={k}: Silhouette={silhouette_scores[i]:.4f}, DB={db_scores[i]:.4f}, WCSS={wcss[i]:.0f}")

In [ ]:
# Chọn số cluster tối ưu dựa trên kết quả trên
# Mặc định k=4 (phổ biến cho RFM), thay đổi nếu cần
OPTIMAL_K = 4
print(f"Số cluster được chọn: k = {OPTIMAL_K}")
print(f"  → Silhouette Score: {silhouette_scores[OPTIMAL_K-2]:.4f}")
print(f"  → Davies-Bouldin:   {db_scores[OPTIMAL_K-2]:.4f}")

## 8. K-Means Clustering

In [ ]:
# =========================================================
# K-MEANS CLUSTERING
# =========================================================
kmeans = KMeans(n_clusters=OPTIMAL_K, random_state=42, n_init=10)
rfm["KMeans_Label"] = kmeans.fit_predict(kmeans_scaled)

# Validate final cluster
final_sil = silhouette_score(kmeans_scaled, rfm["KMeans_Label"])
final_db  = davies_bouldin_score(kmeans_scaled, rfm["KMeans_Label"])
print(f"Final model — k={OPTIMAL_K}")
print(f"  Silhouette Score  : {final_sil:.4f}  (range -1 to 1, closer to 1 is better)")
print(f"  Davies-Bouldin    : {final_db:.4f}   (lower is better)")

## 9. Cluster Profiling & Dynamic Naming

> **[FIX v2]** Tên cluster được gán **tự động** dựa trên profile thực tế (mean Recency, Frequency, Monetary) sau mỗi lần fit — không hard-code label số.

In [ ]:
# =========================================================
# CLUSTER PROFILE ANALYSIS
# =========================================================
cluster_profile = rfm.groupby("KMeans_Label").agg(
    Count=("CustomerID", "count"),
    Recency_Mean=("Recency", "mean"),
    Frequency_Mean=("Frequency", "mean"),
    Monetary_Mean=("Monetary", "mean"),
    Monetary_Median=("Monetary", "median"),
    Monetary_Sum=("Monetary", "sum")
).round(2)

cluster_profile["Revenue_Share_%"] = (
    cluster_profile["Monetary_Sum"] /
    cluster_profile["Monetary_Sum"].sum() * 100
).round(1)

print("Raw Cluster Profile:")
display(cluster_profile)

In [ ]:
# =========================================================
# [FIX] DYNAMIC CLUSTER NAMING
# Gán tên dựa trên rank của từng cluster theo R, F, M
# Không dùng số cố định như trước
# =========================================================
def assign_cluster_names(profile_df):
    """
    Gán tên cluster tự động dựa trên mean R, F, M.
    VIP    : Recency thấp nhất + Frequency cao nhất + Monetary cao nhất
    Loyal  : Frequency và Monetary cao, Recency trung bình
    At Risk: Recency cao (lâu không mua), F và M trung bình+
    Standard: Recency cao + Frequency thấp + Monetary thấp
    """
    df = profile_df.copy()

    # Tạo composite score: thấp Recency tốt → đảo dấu
    df["composite"] = (
        - df["Recency_Mean"].rank()        # thấp = tốt
        + df["Frequency_Mean"].rank()      # cao = tốt
        + df["Monetary_Mean"].rank()       # cao = tốt
    )

    sorted_idx = df["composite"].sort_values(ascending=False).index

    n = len(sorted_idx)
    name_map = {}

    if n == 4:
        names = ["VIP Customers", "Loyal Customers",
                 "Regular Customers", "At Risk Customers"]
    elif n == 3:
        names = ["VIP Customers", "Loyal Customers", "At Risk Customers"]
    elif n == 5:
        names = ["VIP Customers", "Loyal Customers", "Regular Customers",
                 "At Risk Customers", "Lost Customers"]
    else:
        names = [f"Cluster {i+1}" for i in range(n)]

    for label, name in zip(sorted_idx, names):
        name_map[label] = name

    return name_map

cluster_names = assign_cluster_names(cluster_profile)

print("Dynamic Cluster Name Mapping:")
for label, name in cluster_names.items():
    row = cluster_profile.loc[label]
    print(f"  Label {label} → '{name}'")
    print(f"    Recency={row['Recency_Mean']:.0f}d, "
          f"Freq={row['Frequency_Mean']:.1f}, "
          f"Monetary={row['Monetary_Mean']:,.0f}, "
          f"Count={row['Count']:,} ({row['Revenue_Share_%']}% revenue)")

rfm["Cluster_Name"] = rfm["KMeans_Label"].map(cluster_names)

## 10. Visualizations

In [ ]:
# =========================================================
# CUSTOMER DISTRIBUTION BY CLUSTER
# =========================================================
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cluster_counts = rfm["Cluster_Name"].value_counts()

# Bar chart
bars = axes[0].bar(
    cluster_counts.index,
    cluster_counts.values,
    color=["#2196F3", "#4CAF50", "#FF9800", "#F44336", "#9C27B0"][:len(cluster_counts)],
    edgecolor='white', alpha=0.85
)
axes[0].set_title("Customer Distribution by Cluster", fontsize=12)
axes[0].set_xlabel("Cluster")
axes[0].set_ylabel("Number of Customers")
axes[0].tick_params(axis='x', rotation=15)

for bar in bars:
    axes[0].text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height() + 0.3,
        f"{int(bar.get_height()):,}",
        ha='center', va='bottom', fontsize=9
    )

# Pie
axes[1].pie(cluster_counts.values, labels=cluster_counts.index,
            autopct="%1.1f%%", startangle=140)
axes[1].set_title("Customer Share by Cluster", fontsize=12)

plt.tight_layout()
plt.show()

In [ ]:
# =========================================================
# SCATTER PLOT: FREQUENCY vs MONETARY
# =========================================================
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=rfm,
    x="Frequency",
    y="Monetary",
    hue="Cluster_Name",
    alpha=0.6,
    s=50,
    palette="Set1"
)

plt.title("Customer Segmentation — Frequency vs Monetary", fontsize=13)
plt.xlabel("Frequency (số giao dịch)")
plt.ylabel("Monetary (tổng doanh thu)")
plt.legend(title="Cluster", bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# =========================================================
# SCATTER PLOT: RECENCY vs MONETARY
# =========================================================
plt.figure(figsize=(10, 6))

sns.scatterplot(
    data=rfm,
    x="Recency",
    y="Monetary",
    hue="Cluster_Name",
    alpha=0.6,
    s=50,
    palette="Set1"
)

plt.title("Customer Segmentation — Recency vs Monetary", fontsize=13)
plt.xlabel("Recency (ngày kể từ lần mua cuối)")
plt.ylabel("Monetary (tổng doanh thu)")
plt.legend(title="Cluster", bbox_to_anchor=(1.01, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# =========================================================
# HEATMAP CLUSTER PROFILES
# =========================================================
# Normalize từng cột về 0-1 để so sánh trực quan
profile_display = cluster_profile[["Recency_Mean","Frequency_Mean","Monetary_Mean"]].copy()

# Recency: đảo ngược (thấp = tốt)
profile_display["Recency_Mean"] = profile_display["Recency_Mean"].max() - profile_display["Recency_Mean"]

profile_norm = (profile_display - profile_display.min()) / (
    profile_display.max() - profile_display.min()
)
profile_norm.index = [cluster_names.get(i, f"Cluster {i}") for i in profile_norm.index]
profile_norm.columns = ["Recency (inverted)", "Frequency", "Monetary"]

plt.figure(figsize=(8, 4))
sns.heatmap(
    profile_norm,
    annot=True,
    fmt=".2f",
    cmap="YlGnBu",
    linewidths=0.5,
    vmin=0, vmax=1
)
plt.title("Cluster Profile Heatmap (normalized, higher = better)", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# =========================================================
# REVENUE CONTRIBUTION BY CLUSTER
# =========================================================
revenue_by_cluster = rfm.groupby("Cluster_Name")["Monetary"].sum().sort_values(ascending=False)
revenue_pct = (revenue_by_cluster / revenue_by_cluster.sum() * 100).round(1)

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.barh(
    revenue_by_cluster.index,
    revenue_by_cluster.values,
    color=["#2196F3", "#4CAF50", "#FF9800", "#F44336", "#9C27B0"][:len(revenue_by_cluster)],
    edgecolor='white', alpha=0.85
)
for bar, pct in zip(bars, revenue_pct.values):
    ax.text(
        bar.get_width() * 1.01, bar.get_y() + bar.get_height()/2,
        f"{pct}% of revenue",
        va='center', fontsize=9
    )
ax.set_title("Revenue Contribution by Cluster", fontsize=12)
ax.set_xlabel("Total Revenue")
ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:,.0f}"))
plt.tight_layout()
plt.show()

## 11. Business Action Recommendations

> **[ADD v2]** Mỗi phân khúc có chiến lược marketing và retention cụ thể.

In [ ]:
# =========================================================
# [ADD] BUSINESS ACTION FRAMEWORK
# =========================================================
business_actions = {
    "VIP Customers": {
        "description": "Mua gần đây, thường xuyên, chi nhiều tiền nhất",
        "strategy": "Loyalty Program & Exclusive Benefits",
        "actions": [
            "Chương trình VIP membership với ưu đãi độc quyền",
            "Early access sản phẩm mới / limited edition",
            "Dedicated customer success manager",
            "Personalized thank-you gifts theo anniversary"
        ],
        "kpi": "Duy trì Purchase Frequency, tăng AOV (Average Order Value)"
    },
    "Loyal Customers": {
        "description": "Mua thường xuyên và chi nhiều, nhưng Recency trung bình",
        "strategy": "Upsell & Cross-sell để nâng lên VIP",
        "actions": [
            "Chương trình tích điểm / rewards với ngưỡng VIP rõ ràng",
            "Bundle offer với sản phẩm bổ sung",
            "Referral program: giới thiệu bạn bè nhận điểm thưởng",
            "Email campaign nhắc nhở mua lại định kỳ"
        ],
        "kpi": "Tăng Purchase Frequency, giảm Recency"
    },
    "Regular Customers": {
        "description": "Mua ở mức trung bình, không nổi bật về mặt nào",
        "strategy": "Nurturing & Education để tăng engagement",
        "actions": [
            "Content marketing giáo dục về sản phẩm",
            "Discount theo ngưỡng: mua thêm X để nhận ưu đãi",
            "Seasonal promotions theo dịp",
            "Win-back email nếu không mua trong 60 ngày"
        ],
        "kpi": "Tăng Frequency, tăng Monetary per transaction"
    },
    "At Risk Customers": {
        "description": "Từng mua nhiều nhưng lâu không quay lại",
        "strategy": "Win-back Campaign khẩn cấp",
        "actions": [
            "Win-back email sequence: 'We miss you' + discount mạnh (15-25%)",
            "Survey lý do không mua lại",
            "Reactivation offer với thời hạn ngắn (7 ngày)",
            "Phone call hoặc SMS nếu CLV cao"
        ],
        "kpi": "Tỷ lệ reactivation trong 30 ngày"
    },
    "Hibernating": {
        "description": "Ít mua, lâu không quay lại — gần như đã churn",
        "strategy": "Last-chance Campaign hoặc Accept Churn",
        "actions": [
            "One-time deep discount (30%+) với thời hạn cứng",
            "Nếu không phản hồi: di chuyển sang danh sách suppression",
            "Giảm tần suất email để giữ deliverability"
        ],
        "kpi": "Cost per reactivation vs CLV"
    },
    "New Customers": {
        "description": "Mới mua lần đầu, chưa thiết lập habit",
        "strategy": "Onboarding & First-purchase Follow-up",
        "actions": [
            "Welcome email series (3 email trong 14 ngày đầu)",
            "Hướng dẫn sử dụng sản phẩm / tips",
            "Second-purchase incentive (discount 10% cho lần mua tiếp theo)",
            "Khuyến khích review/feedback"
        ],
        "kpi": "Tỷ lệ second purchase trong 30 ngày"
    },
    "Potential Loyalists": {
        "description": "Mua gần đây, có tần suất trung bình — tiềm năng trở thành Loyal",
        "strategy": "Cultivation — nudge lên Loyal trong 60 ngày",
        "actions": [
            "Chương trình challenge: mua 3 lần trong 60 ngày nhận quà",
            "Personalized product recommendations",
            "Social proof: bestsellers, reviews"
        ],
        "kpi": "Tỷ lệ chuyển sang Loyal sau 60 ngày"
    },
    "Champions": {
        "description": "Điểm R/F/M đều cao nhất — top tier",
        "strategy": "Ambassador Program",
        "actions": [
            "Mời tham gia Brand Ambassador",
            "Ưu tiên nhận beta test sản phẩm",
            "Co-creation: lấy ý kiến về sản phẩm mới"
        ],
        "kpi": "NPS score, referral conversion rate"
    },
    "Standard": {
        "description": "Điểm RFM thấp, không đủ tiêu chí các nhóm trên",
        "strategy": "Broad nurturing với chi phí thấp",
        "actions": [
            "Mass email với offer chung",
            "Seasonal sale",
            "Không đầu tư campaign riêng biệt"
        ],
        "kpi": "Revenue per email sent"
    }
}

print("=" * 65)
print("BUSINESS ACTION RECOMMENDATIONS PER SEGMENT")
print("=" * 65)

# Lấy các segment thực tế trong data
actual_kmeans_segments = rfm["Cluster_Name"].unique()
actual_heuristic_segments = rfm["Heuristic_Segment"].unique()
all_segments = set(list(actual_kmeans_segments) + list(actual_heuristic_segments))

for seg in sorted(all_segments):
    if seg in business_actions:
        info = business_actions[seg]
        count = len(rfm[rfm["Cluster_Name"] == seg]) if seg in actual_kmeans_segments else len(rfm[rfm["Heuristic_Segment"] == seg])
        print(f"\n📌 {seg} ({count:,} customers)")
        print(f"   {info['description']}")
        print(f"   Strategy : {info['strategy']}")
        print(f"   KPI      : {info['kpi']}")
        print("   Actions:")
        for a in info['actions']:
            print(f"     • {a}")

print("\n" + "=" * 65)

## 12. Summary Table

In [ ]:
# =========================================================
# FINAL SUMMARY TABLE
# =========================================================
summary = rfm.groupby("Cluster_Name").agg(
    Customers=("CustomerID", "count"),
    Recency_Avg=("Recency", "mean"),
    Frequency_Avg=("Frequency", "mean"),
    Monetary_Avg=("Monetary", "mean"),
    Total_Revenue=("Monetary", "sum")
).round(2)

summary["Customer_%"] = (summary["Customers"] / summary["Customers"].sum() * 100).round(1)
summary["Revenue_%"] = (summary["Total_Revenue"] / summary["Total_Revenue"].sum() * 100).round(1)
summary = summary.sort_values("Total_Revenue", ascending=False)

print("FINAL CLUSTER SUMMARY")
display(summary[["Customers","Customer_%","Recency_Avg","Frequency_Avg",
                 "Monetary_Avg","Total_Revenue","Revenue_%"]])

## 13. Export Results

> **[ADD v2]** Export `rfm` với đầy đủ labels để dùng cho CLV Prediction và Churn Prediction notebooks.

In [ ]:
# =========================================================
# [ADD] EXPORT KẾT QUẢ
# =========================================================
output_cols = [
    "CustomerID",
    "Recency", "Frequency", "Monetary",
    "R_Score", "F_Score", "M_Score", "RFM_Score",
    "KMeans_Label", "Cluster_Name",
    "Heuristic_Segment"
]

rfm_export = rfm[output_cols].copy()

# Lưu vào cùng thư mục Dataset để dùng cho notebooks khác
import os
output_path = "Dataset/Cleaned_data/rfm_segments.csv"
os.makedirs(os.path.dirname(output_path), exist_ok=True)
rfm_export.to_csv(output_path, index=False)

print(f"✓ Exported {len(rfm_export):,} rows to '{output_path}'")
print(f"\nColumn list: {list(rfm_export.columns)}")
print(f"\nUsage in CLV/Churn notebooks:")
print(f"  segments = pd.read_csv('{output_path}')")
print(f"  df = df.merge(segments[['CustomerID','KMeans_Label','Cluster_Name']], on='CustomerID', how='left')")

rfm_export.head()

---
## Summary of Changes

| # | Vấn đề cũ | Sửa trong v2 |
|---|---|---|
| 1 | Monetary dùng cột `Invoice` (mã đơn hàng) | Auto-detect cột doanh thu thực (`total_revenue`) |
| 2 | Cluster naming hard-coded theo label số | Dynamic naming dựa trên composite score của profile |
| 3 | Heuristic segment chỉ dùng tổng RFM Score | Logic theo từng chiều R, F, M riêng biệt với 7 nhóm |
| 4 | Không có cluster validation | Silhouette Score + Davies-Bouldin Index |
| 5 | Scale Monetary raw (phân phối lệch phải) | Log-transform trước StandardScaler |
| 6 | Không kiểm tra outlier | IQR boxplot + thống kê outlier |
| 7 | Không có actionable insights | Business action framework per segment |
| 8 | Không export để dùng tiếp | Export `rfm_segments.csv` cho CLV & Churn notebooks |